# 02 · Data Overview & Variable Selection (NHANES 2015–2018)

En este notebook se completa la fase de **Data Understanding** de CRISP-DM, enfocándose en:

1. Explorar la estructura general de los archivos NHANES utilizados:
   - DEMO (demografía)
   - BMX (medidas corporales)
   - BPX (presión arterial)
   - PAQ (actividad física)
2. Documentar la **selección de variables** que se utilizarán en el proyecto.
3. Generar datasets intermedios con solo las columnas relevantes, separados por ciclo:
   - `set1_*` para 2015–2016 (`*_I`)
   - `set2_*` para 2017–2018 (`*_J`)

Esta selección temprana de columnas busca:
- Reducir complejidad en las etapas posteriores de EDA y preparación.
- Mantener el foco en variables alineadas con las hipótesis planteadas en el Notebook 01.


In [1]:
# 1. Importar librerías y definir rutas base
!unzip data.zip

import pandas as pd
from pathlib import Path

# Directorio base del proyecto
BASE_PATH = Path(".")

# Rutas a los datos brutos de NHANES
DATA_15_16 = BASE_PATH / "data" / "15-16"  # DEMO_I, BMX_I, BPX_I, PAQ_I
DATA_17_18 = BASE_PATH / "data" / "17-18"  # DEMO_J, BMX_J, BPX_J, PAQ_J

print("Ruta 2015-2016:", DATA_15_16.resolve())
print("Ruta 2017-2018:", DATA_17_18.resolve())


Archive:  data.zip
   creating: data/
   creating: data/15-16/
  inflating: data/15-16/BMX_I.csv    
  inflating: data/15-16/BPX_I.csv    
  inflating: data/15-16/DEMO_I.csv   
  inflating: data/15-16/MCQ_I.csv    
  inflating: data/15-16/PAQ_I.csv    
   creating: data/17-18/
  inflating: data/17-18/BMX_J.csv    
  inflating: data/17-18/BPX_J.csv    
  inflating: data/17-18/DEMO_J.csv   
  inflating: data/17-18/MCQ_J.csv    
  inflating: data/17-18/PAQ_J.csv    
Ruta 2015-2016: /content/data/15-16
Ruta 2017-2018: /content/data/17-18


In [3]:
# 2. Vista general de columnas por archivo (carga ligera con nrows)

files_15_16 = {
    "DEMO_I": DATA_15_16 / "DEMO_I.csv",
    "BMX_I":  DATA_15_16 / "BMX_I.csv",
    "BPX_I":  DATA_15_16 / "BPX_I.csv",
    "PAQ_I":  DATA_15_16 / "PAQ_I.csv",
}

files_17_18 = {
    "DEMO_J": DATA_17_18 / "DEMO_J.csv",
    "BMX_J":  DATA_17_18 / "BMX_J.csv",
    "BPX_J":  DATA_17_18 / "BPX_J.csv",
    "PAQ_J":  DATA_17_18 / "PAQ_J.csv",
}

def show_columns_sample(path: Path, nrows: int = 5):
    """Carga solo algunas filas para inspeccionar columnas."""
    df_sample = pd.read_csv(path, nrows=nrows)
    print(f"Archivo: {path.name}")
    print(f"  Número de columnas: {len(df_sample.columns)}")
    print(f"  Ejemplo de columnas: {list(df_sample.columns)[:10]} ...\n")
    return df_sample.columns


print("=== Ciclo 2015–2016 (sufijo _I) ===")
cols_15_16 = {}
for name, path in files_15_16.items():
    cols_15_16[name] = show_columns_sample(path)

print("=== Ciclo 2017–2018 (sufijo _J) ===")
cols_17_18 = {}
for name, path in files_17_18.items():
    cols_17_18[name] = show_columns_sample(path)


=== Ciclo 2015–2016 (sufijo _I) ===
Archivo: DEMO_I.csv
  Número de columnas: 47
  Ejemplo de columnas: ['SEQN', 'SDDSRVYR', 'RIDSTATR', 'RIAGENDR', 'RIDAGEYR', 'RIDAGEMN', 'RIDRETH1', 'RIDRETH3', 'RIDEXMON', 'RIDEXAGM'] ...

Archivo: BMX_I.csv
  Número de columnas: 26
  Ejemplo de columnas: ['SEQN', 'BMDSTATS', 'BMXWT', 'BMIWT', 'BMXRECUM', 'BMIRECUM', 'BMXHEAD', 'BMIHEAD', 'BMXHT', 'BMIHT'] ...

Archivo: BPX_I.csv
  Número de columnas: 21
  Ejemplo de columnas: ['SEQN', 'PEASCCT1', 'BPXCHR', 'BPAARM', 'BPACSZ', 'BPXPLS', 'BPXPULS', 'BPXPTY', 'BPXML1', 'BPXSY1'] ...

Archivo: PAQ_I.csv
  Número de columnas: 94
  Ejemplo de columnas: ['SEQN', 'PAQ605', 'PAQ610', 'PAD615', 'PAQ620', 'PAQ625', 'PAD630', 'PAQ635', 'PAQ640', 'PAD645'] ...

=== Ciclo 2017–2018 (sufijo _J) ===
Archivo: DEMO_J.csv
  Número de columnas: 46
  Ejemplo de columnas: ['SEQN', 'SDDSRVYR', 'RIDSTATR', 'RIAGENDR', 'RIDAGEYR', 'RIDAGEMN', 'RIDRETH1', 'RIDRETH3', 'RIDEXMON', 'RIDEXAGM'] ...

Archivo: BMX_J.csv
  Número 

### 2. Explicación

Objetivo de esta celda:

1. Tener una **vista general** de:
   - Cuántas columnas tiene cada archivo DEMO/BMX/BPX/PAQ.
   - Ejemplos de nombres de columnas.
2. Hacerlo de forma **ligera**, leyendo solo unas pocas filas (`nrows=5`) sin filtros de `usecols`.


In [4]:
# 3. Definición formal del subset de columnas a utilizar

demo_cols = [
    "SEQN",      # Identificador del participante
    "RIDAGEYR",  # Edad
    "RIAGENDR",  # Sexo
    "RIDRETH3",  # Raza/Etnia
    "DMDEDUC2",  # Nivel de Educación
    "INDFMPIR",  # Índice de Ingreso/Pobreza
    # Opcionalmente se podríaañadir "RIDSTATR" si se quiere filtrar "examinados en MEC" más adelante
]

bmx_cols = [
    "SEQN",
    "BMXWT",     # Peso
    "BMXHT",     # Talla
    "BMXBMI",    # IMC
    "BMXWAIST",  # Circunferencia de cintura
]

bpx_cols = [
    "SEQN",
    "BPXSY1", "BPXSY2", "BPXSY3", "BPXSY4",
    "BPXDI1", "BPXDI2", "BPXDI3", "BPXDI4",
]

paq_cols = [
    "SEQN",
    # Transporte
    "PAQ635", "PAQ640", "PAD645",
    # Recreación vigorosa
    "PAQ650", "PAQ655", "PAD660",
    # Recreación moderada
    "PAQ665", "PAQ670", "PAD675",
    # Actividad laboral
    "PAQ620", "PAD630",
    # Sedentarismo
    "PAD680",
]

demo_cols, bmx_cols, bpx_cols, paq_cols


(['SEQN', 'RIDAGEYR', 'RIAGENDR', 'RIDRETH3', 'DMDEDUC2', 'INDFMPIR'],
 ['SEQN', 'BMXWT', 'BMXHT', 'BMXBMI', 'BMXWAIST'],
 ['SEQN',
  'BPXSY1',
  'BPXSY2',
  'BPXSY3',
  'BPXSY4',
  'BPXDI1',
  'BPXDI2',
  'BPXDI3',
  'BPXDI4'],
 ['SEQN',
  'PAQ635',
  'PAQ640',
  'PAD645',
  'PAQ650',
  'PAQ655',
  'PAD660',
  'PAQ665',
  'PAQ670',
  'PAD675',
  'PAQ620',
  'PAD630',
  'PAD680'])

### 3. Explicación

En esta celda se definen explícitamente las **listas de columnas** que se utilizarán en el resto del proyecto:

1. `demo_cols`  
   - Variables demográficas clave:
     1. `SEQN`: identificador único (para joins).
     2. `RIDAGEYR`: edad (fundamental para riesgo cardiovascular).
     3. `RIAGENDR`: sexo.
     4. `RIDRETH3`: raza/etnia.
     5. `DMDEDUC2`: nivel de educación (proxy de nivel socioeconómico).
     6. `INDFMPIR`: índice ingreso/pobreza (medida continua de situación socioeconómica).

2. `bmx_cols`  
   - Medidas antropométricas asociadas a obesidad y riesgo cardiometabólico:
     1. `BMXWT`: peso.
     2. `BMXHT`: talla.
     3. `BMXBMI`: IMC.
     4. `BMXWAIST`: circunferencia de cintura.

3. `bpx_cols`  
   - Lecturas de presión arterial:
     - Sistólica: `BPXSY1–4`.
     - Diastólica: `BPXDI1–4`.
   - Se usarán **solo** para derivar:
     - `SBP_mean`, `DBP_mean` y `HTN_label`.
   - No se usarán como *features* directas, para evitar *data leakage*.

4. `paq_cols`  
   - Bloques de actividad física y sedentarismo:
     1. Transporte activo (`PAQ635`, `PAQ640`, `PAD645`).
     2. Recreación vigorosa (`PAQ650`, `PAQ655`, `PAD660`).
     3. Recreación moderada (`PAQ665`, `PAQ670`, `PAD675`).
     4. Actividad laboral (`PAQ620`, `PAD630`).
     5. Sedentarismo (`PAD680`).

Estas listas formalizan la **selección temprana de variables**, alineada con las hipótesis de presión arterial e hipertensión.


In [5]:
# 4. Tabla resumen de variables seleccionadas y justificación

rows = []

# DEMO
for col in demo_cols:
    if col == "SEQN":
        motivo = "Identificador único, necesario para unir DEMO/BMX/BPX/PAQ."
    elif col == "RIDAGEYR":
        motivo = "Edad: factor de riesgo mayor para presión arterial e HTA."
    elif col == "RIAGENDR":
        motivo = "Sexo: permite capturar diferencias de riesgo entre hombres y mujeres."
    elif col == "RIDRETH3":
        motivo = "Raza/Etnia: ajuste por diferencias poblacionales conocidas en HTA."
    elif col == "DMDEDUC2":
        motivo = "Nivel educacional: proxy de nivel socioeconómico y estilo de vida."
    elif col == "INDFMPIR":
        motivo = "Índice ingreso/pobreza: medida continua de situación socioeconómica."
    else:
        motivo = ""
    rows.append(["DEMO", col, True, motivo])

# BMX
for col in bmx_cols:
    if col == "SEQN":
        motivo = "Identificador para unir con DEMO/BPX/PAQ."
    elif col == "BMXWT":
        motivo = "Peso: componente básico de antropometría y cálculo de IMC."
    elif col == "BMXHT":
        motivo = "Talla: necesaria para interpretar peso e IMC."
    elif col == "BMXBMI":
        motivo = "IMC: indicador estándar de obesidad."
    elif col == "BMXWAIST":
        motivo = "Cintura: obesidad abdominal, fuertemente asociada a HTA."
    else:
        motivo = ""
    rows.append(["BMX", col, True, motivo])

# BPX
for col in bpx_cols:
    if col == "SEQN":
        motivo = "Identificador para unir lecturas de presión con DEMO/BMX/PAQ."
    else:
        motivo = "Lectura cruda de presión arterial (sistólica/diastólica); se usará para derivar SBP_mean/DBP_mean/HTN_label."
    rows.append(["BPX", col, True, motivo])

# PAQ
for col in paq_cols:
    if col == "SEQN":
        motivo = "Identificador para unir cuestionario de actividad física con el resto."
    elif col in ["PAQ635", "PAQ640", "PAD645"]:
        motivo = "Transporte activo (caminar/bici): componente de actividad física diaria."
    elif col in ["PAQ650", "PAQ655", "PAD660"]:
        motivo = "Actividad recreativa vigorosa: intensidad alta, relevante para control de presión."
    elif col in ["PAQ665", "PAQ670", "PAD675"]:
        motivo = "Actividad recreativa moderada: contribuye al gasto energético global."
    elif col in ["PAQ620", "PAD630"]:
        motivo = "Actividad moderada laboral: componente importante de actividad física total."
    elif col == "PAD680":
        motivo = "Tiempo sedentario diario: predictor clave de riesgo cardiometabólico."
    else:
        motivo = ""
    rows.append(["PAQ", col, True, motivo])

selected_vars = pd.DataFrame(
    rows,
    columns=["tabla", "columna", "usar_en_pipeline", "motivo"]
)

selected_vars


,tabla,columna,usar_en_pipeline,motivo
0,DEMO,SEQN,True,"Identificador único, necesario para unir DEMO/..."
1,DEMO,RIDAGEYR,True,Edad: factor de riesgo mayor para presión arte...
2,DEMO,RIAGENDR,True,Sexo: permite capturar diferencias de riesgo e...
3,DEMO,RIDRETH3,True,Raza/Etnia: ajuste por diferencias poblacional...
4,DEMO,DMDEDUC2,True,Nivel educacional: proxy de nivel socioeconómi...
5,DEMO,INDFMPIR,True,Índice ingreso/pobreza: medida continua de sit...
6,BMX,SEQN,True,Identificador para unir con DEMO/BPX/PAQ.
7,BMX,BMXWT,True,Peso: componente básico de antropometría y cál...
8,BMX,BMXHT,True,Talla: necesaria para interpretar peso e IMC.
9,BMX,BMXBMI,True,IMC: indicador estándar de obesidad.


### 4. Explicación

Esta celda construye una **tabla resumen** de las variables seleccionadas:

- Columnas:
  1. `tabla`: origen de la variable (`DEMO`, `BMX`, `BPX`, `PAQ`).
  2. `columna`: nombre de la variable en NHANES.
  3. `usar_en_pipeline`: indicador explícito (True) de inclusión.
  4. `motivo`: justificación breve de por qué se incluye.

Propósito:
* Mostrar que la decisión se basa en:
   - Relevancia clínica/epidemiológica para hipertensión.
   - Rol como identificador (SEQN).
   - Contribución a la actividad física o la antropometría.

Observación importante:

- Aquí se listan solo las variables **incluidas**.
- Se asume que todas las demás columnas de cada archivo existen pero se **excluyen** por:
  - No estar directamente relacionadas con los objetivos del proyecto.
  - Aumentar complejidad sin un beneficio evidente para las hipótesis planteadas.


In [6]:
# 5. Carga de los datasets filtrando solo las columnas seleccionadas (set1_* y set2_*)

# DEMO
set1_demographic = pd.read_csv(DATA_15_16 / "DEMO_I.csv", usecols=demo_cols)
set2_demographic = pd.read_csv(DATA_17_18 / "DEMO_J.csv", usecols=demo_cols)

# BMX
set1_body_measurements = pd.read_csv(DATA_15_16 / "BMX_I.csv", usecols=bmx_cols)
set2_body_measurements = pd.read_csv(DATA_17_18 / "BMX_J.csv", usecols=bmx_cols)

# BPX
set1_blood_pressure = pd.read_csv(DATA_15_16 / "BPX_I.csv", usecols=bpx_cols)
set2_blood_pressure = pd.read_csv(DATA_17_18 / "BPX_J.csv", usecols=bpx_cols)

# PAQ
set1_questionnaire = pd.read_csv(DATA_15_16 / "PAQ_I.csv", usecols=paq_cols)
set2_questionnaire = pd.read_csv(DATA_17_18 / "PAQ_J.csv", usecols=paq_cols)

print("set1_demographic:", set1_demographic.shape)
print("set2_demographic:", set2_demographic.shape)
print("set1_body_measurements:", set1_body_measurements.shape)
print("set2_body_measurements:", set2_body_measurements.shape)
print("set1_blood_pressure:", set1_blood_pressure.shape)
print("set2_blood_pressure:", set2_blood_pressure.shape)
print("set1_questionnaire:", set1_questionnaire.shape)
print("set2_questionnaire:", set2_questionnaire.shape)


set1_demographic: (9971, 6)
set2_demographic: (9254, 6)
set1_body_measurements: (9544, 5)
set2_body_measurements: (8704, 5)
set1_blood_pressure: (9544, 9)
set2_blood_pressure: (8704, 9)
set1_questionnaire: (9255, 13)
set2_questionnaire: (5856, 13)


In [7]:
# 6. Vista rápida de los dataframes filtrados (head)

datasets = {
    "set1_demographic": set1_demographic,
    "set2_demographic": set2_demographic,
    "set1_body_measurements": set1_body_measurements,
    "set2_body_measurements": set2_body_measurements,
    "set1_blood_pressure": set1_blood_pressure,
    "set2_blood_pressure": set2_blood_pressure,
    "set1_questionnaire": set1_questionnaire,
    "set2_questionnaire": set2_questionnaire,
}

for name, df in datasets.items():
    print(f"=== {name} ===")
    display(df.head())
    print()


=== set1_demographic ===


,SEQN,RIAGENDR,RIDAGEYR,RIDRETH3,DMDEDUC2,INDFMPIR
0,83732.0,1.0,62.0,3.0,5.0,4.39
1,83733.0,1.0,53.0,3.0,3.0,1.32
2,83734.0,1.0,78.0,3.0,3.0,1.51
3,83735.0,2.0,56.0,3.0,5.0,5.00
4,83736.0,2.0,42.0,4.0,4.0,1.23



=== set2_demographic ===


,SEQN,RIAGENDR,RIDAGEYR,RIDRETH3,DMDEDUC2,INDFMPIR
0,93703.0,2.0,2.0,6.0,NaN,5.00
1,93704.0,1.0,2.0,3.0,NaN,5.00
2,93705.0,2.0,66.0,4.0,2.0,0.82
3,93706.0,1.0,18.0,6.0,NaN,NaN
4,93707.0,1.0,13.0,7.0,NaN,1.88



=== set1_body_measurements ===


...

,SEQN,BPXSY1,BPXDI1,BPXSY2,BPXDI2,BPXSY3,BPXDI3,BPXSY4,BPXDI4
0,93703.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,93704.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,93705.0,NaN,NaN,NaN,NaN,202.0,62.0,198.0,74.0
3,93706.0,112.0,74.0,114.0,70.0,108.0,76.0,NaN,NaN
4,93707.0,128.0,38.0,128.0,46.0,128.0,58.0,NaN,NaN



=== set1_questionnaire ===


,SEQN,PAQ620,PAD630,PAQ635,PAQ640,PAD645,PAQ650,PAQ655,PAD660,PAQ665,PAQ670,PAD675,PAD680
0,83732.0,1.0,10.0,2.0,NaN,NaN,2.0,NaN,NaN,1.0,6.0,30.0,480.0
1,83733.0,2.0,NaN,2.0,NaN,NaN,2.0,NaN,NaN,2.0,NaN,NaN,300.0
2,83734.0,1.0,240.0,2.0,NaN,NaN,2.0,NaN,NaN,2.0,NaN,NaN,480.0
3,83735.0,1.0,90.0,2.0,NaN,NaN,2.0,NaN,NaN,2.0,NaN,NaN,480.0
4,83736.0,1.0,480.0,2.0,NaN,NaN,2.0,NaN,NaN,2.0,NaN,NaN,540.0



=== set2_questionnaire ===


,SEQN,PAQ620,PAD630,PAQ635,PAQ640,PAD645,PAQ650,PAQ655,PAD660,PAQ665,PAQ670,PAD675,PAD680
0,93705.0,2.0,NaN,2.0,NaN,NaN,2.0,NaN,NaN,1.0,2.0,60.0,300.0
1,93706.0,2.0,NaN,1.0,5.0,45.0,2.0,NaN,NaN,1.0,2.0,30.0,240.0
2,93708.0,2.0,NaN,2.0,NaN,NaN,2.0,NaN,NaN,1.0,5.0,30.0,120.0
3,93709.0,1.0,180.0,2.0,NaN,NaN,2.0,NaN,NaN,2.0,NaN,NaN,600.0
4,93711.0,2.0,NaN,1.0,5.0,60.0,1.0,4.0,60.0,1.0,2.0,30.0,420.0


### 6. Explicación

En esta celda se muestran las primeras filas de cada dataframe:

- `set1_*` corresponden a 2015–2016.
- `set2_*` corresponden a 2017–2018.

Objetivos:

1. Verificar que:
   - Los nombres de las columnas son los esperados.
   - `SEQN` está presente en todas las tablas.
2. Tener una “evidencia visual” de la estructura que se usará en el pipeline de preparación y modelamiento.

Este paso es útil para detectar errores tempranos, como:
- Columnas mal tipeadas.
- CSV vacíos o con un número inesperado de filas.


In [8]:
# 7. Guardar los datasets filtrados como CSV intermedios (data/processed)

PROCESSED_DIR = BASE_PATH / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

set1_demographic.to_csv(PROCESSED_DIR / "set1_demographic.csv", index=False)
set2_demographic.to_csv(PROCESSED_DIR / "set2_demographic.csv", index=False)

set1_body_measurements.to_csv(PROCESSED_DIR / "set1_body_measurements.csv", index=False)
set2_body_measurements.to_csv(PROCESSED_DIR / "set2_body_measurements.csv", index=False)

set1_blood_pressure.to_csv(PROCESSED_DIR / "set1_blood_pressure.csv", index=False)
set2_blood_pressure.to_csv(PROCESSED_DIR / "set2_blood_pressure.csv", index=False)

set1_questionnaire.to_csv(PROCESSED_DIR / "set1_questionnaire.csv", index=False)
set2_questionnaire.to_csv(PROCESSED_DIR / "set2_questionnaire.csv", index=False)

print("Archivos intermedios guardados en:", PROCESSED_DIR.resolve())


Archivos intermedios guardados en: /content/data/processed


### 7. Explicación

Esta celda finaliza la fase de **selección de variables** generando archivos intermedios:

1. Se define `PROCESSED_DIR = data/processed` y se crea la carpeta si no existe.
2. Cada dataframe filtrado se guarda como CSV:
   - `set1_demographic.csv`, `set2_demographic.csv`
   - `set1_body_measurements.csv`, `set2_body_measurements.csv`
   - `set1_blood_pressure.csv`, `set2_blood_pressure.csv`
   - `set1_questionnaire.csv`, `set2_questionnaire.csv`

Estos archivos serán:

1. El punto de partida del Notebook 03 (**EDA & Data Preparation**).
2. Las entradas naturales del **Data Catalog** de Kedro (`catalog.yml`), por ejemplo:

   - `set1_demographic` → `data/processed/set1_demographic.csv`
   - etc.

Al separar los datos “brutos” (`*_I`, `*_J`) de estos datos “filtrados”, se mantiene clara la frontera entre:

- La estructura original de NHANES.
- El subset de variables relevante para las hipótesis del proyecto.
